# 06 · Submission (Kaggle, GPU, internet kapalı)

**Bu notebook ne yapar?** Gizli test setindeki her vakayı DICOM'dan okuyup (cache'teki
**aynı** ön işlemeyle), eğitilmiş fold modellerini çalıştırır, fold'ları rank-ortalamasıyla
birleştirir ve `submission.csv` yazar.

**Eski kernel neden 0.500 aldı?** DICOM okunamayınca veya checkpoint uyuşmayınca sessizce
sabit önsel değerler yazıyordu; sabit tahmin her bulgu için AUC = 0.5 demektir.
Bu notebook'ta üç koruma var, herhangi biri tetiklenirse **hata verir** (skor 0.5 almak
yerine gönderimin başarısız olması tercih edilir — hemen fark edersiniz):
1. Checkpoint `strict=True` ile yüklenir; anahtar uyuşmazlığı hata.
2. Okunamayan vaka oranı %2'yi aşarsa hata.
3. Tüm vakalara aynı tahmin yazılmışsa hata.

**Kaggle ayarları:** Accelerator = GPU, Internet = **kapalı** (yarışma kuralı),
Input = yarışma verisi + `rsna-knee-src` + 05'in çıktısı (checkpoint'ler). Süre sınırı 9 saat.

In [ ]:
# Proje kökünü (src/ klasörünü içeren yer) Python yoluna ekle: yerelde ve Kaggle'da çalışır.
import os, sys
from pathlib import Path

def add_project_root():
    cands = [Path.cwd(), Path.cwd().parent]
    if Path("/kaggle/input").exists():
        # os.walk + budama: yarışmanın DICOM klasörlerine girmez (rglob dakikalar sürüyordu)
        for d, dirs, files in os.walk("/kaggle/input"):
            dirs[:] = [x for x in dirs if x not in ("train_series", "test_series")]
            if Path(d).name == "src" and "__init__.py" in files:
                cands.append(Path(d).parent)
    for c in cands:
        if (c / "src" / "__init__.py").exists():
            sys.path.insert(0, str(c))
            return c
    raise FileNotFoundError("src/ bulunamadı")

PROJECT_ROOT = add_project_root()
print("Proje kökü:", PROJECT_ROOT)

## 0. Sıkıştırılmış DICOM çözücüleri

Veri setinde JPEG Lossless ve JPEG 2000 ile sıkıştırılmış seriler var. pydicom bunları ek
paketler olmadan açamaz; Kaggle imajında bu paketler yok. Kurulmazsa o seriler sessizce
atlanır (ilk cache denemesinde yaşadığımız sorun). Wheel'ler `rsna-knee-src` içinde
(`third_party/wheels`), internet gerekmez.

In [ ]:
from pathlib import Path
from src.config import IS_KAGGLE
from src.dicom import ensure_decoders

from src.config import find_input
WHEELS = [p for p in [find_input("third_party/wheels")] if p] if IS_KAGGLE else []
DECODERS = ensure_decoders(WHEELS)
print("DICOM çözücüler:", DECODERS)
if IS_KAGGLE:
    assert DECODERS["libjpeg"] and DECODERS["openjpeg"], "JPEG çözücüleri kurulamadı!"

In [ ]:
import time
from pathlib import Path

import pandas as pd
import torch

from src.cache import CacheConfig
from src.config import DATA_DIR, EXPERIMENTS_DIR, IS_KAGGLE, TARGET_COLS
from src.inference import predict_test, rank_blend, validate_submission
from src.model import load_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt_root = Path("/kaggle/input") if IS_KAGGLE else EXPERIMENTS_DIR
from src.config import find_inputs
CKPTS = find_inputs("model_fold*.pt", str(ckpt_root))
print("Checkpoint'ler:", *CKPTS, sep="\n  ")
if not CKPTS:
    raise FileNotFoundError("Checkpoint bulunamadı: 05'in çıktısını Input olarak ekleyin.")

In [ ]:
models, cache_cfgs = [], set()
for p in CKPTS:
    ck = torch.load(str(p), map_location="cpu", weights_only=False)
    cache_cfgs.add(tuple(sorted(ck["cache_config"].items())))
    models.append(load_checkpoint(p, str(device)))
    print(f"{p.name}: fold {ck['fold']} epoch {ck['epoch']} | {ck['metrics']}")
assert len(cache_cfgs) == 1, "Fold'lar farklı ön işleme ayarlarıyla eğitilmiş!"
CACHE_CFG = CacheConfig(**dict(next(iter(cache_cfgs))))
print("Ön işleme:", CACHE_CFG)

In [ ]:
test = pd.read_csv(DATA_DIR / "test.csv")
series = pd.read_csv(DATA_DIR / "test_series.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
MAX_FAIL = 0.02 if IS_KAGGLE else 1.0  # yerelde 3 vakadan yalnızca 1'inin DICOM'u var
t0 = time.time()
pred, meta = predict_test(models, test, series, DATA_DIR, CACHE_CFG, device,
                          batch_size=2, num_workers=4 if IS_KAGGLE else 0, max_fail_frac=MAX_FAIL)
print(f"Inference: {len(pred)} vaka, {(time.time() - t0) / 60:.1f} dk")
meta.reindex(columns=["StudyInstanceUID", "n_slots", "side", "side_source", "errors"]).head()

## Submission dosyası

Fold'lar zaten rank-ortalaması ile birleşti (değerler 0–1 arası sıra yüzdeleri). AUC
yalnızca sıralamaya baktığı için bu, olasılık ortalamasıyla aynı bilgiyi taşır ve ölçek
farklarına karşı sağlamdır. İleride başka modellerle birleştirmek için `rank_blend`
fonksiyonu aynı şekilde kullanılır.

In [ ]:
sub = sample[["StudyInstanceUID"]].merge(pred, on="StudyInstanceUID", how="left")[["StudyInstanceUID"] + TARGET_COLS]
if IS_KAGGLE:
    validate_submission(sub, sample)
else:
    sub = sub.fillna(0.5)  # yerel deneme: DICOM'u olmayan vakalar
sub.to_csv("submission.csv", index=False)
print("submission.csv yazıldı:", sub.shape)
sub.head()